# 02 website backend

**Owner:** Thomas. Original notebook cells: 93.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

In [ ]:
# THOMAS: Website backend integration interface

import json
import math
import numpy as np
import pandas as pd

MAX_QUESTION_LENGTH = 4000

WEBSITE_ROLE_MAP = {
    "owner": "Owner",
    "receptionist": "Front Desk",
    "front desk": "Front Desk",
    "optician": "Optician"
}

RISK_SCORE_NOTICE = (
    "Model risk scores are uncalibrated classification "
    "scores and are not confirmed probabilities."
)

# Check required upstream components
required_components = [
    "AgentError",
    "ROLE_WORKFLOWS",
    "create_workflow_agent",
    "get_dashboard_records",
    "validate_dashboard_output"
]

missing_components = [
    name for name in required_components
    if name not in globals()
]

if missing_components:

    raise RuntimeError(
        "Missing required components: "
        + ", ".join(missing_components)
    )


# Convert Pandas and NumPy values into native JSON types
def website_json_safe(value):

    if value is pd.NA or value is pd.NaT:
        return None

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, (float, np.floating)):

        number = float(value)

        return number if math.isfinite(number) else None

    if isinstance(value, np.ndarray):
        return website_json_safe(value.tolist())

    if isinstance(value, dict):

        return {
            str(key): website_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):

        return [
            website_json_safe(item)
            for item in value
        ]

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    return value


def json_compatible(value):

    return json.loads(
        json.dumps(
            website_json_safe(value),
            default=str,
            allow_nan=False
        )
    )


def normalize_website_role(role):

    if not isinstance(role, str):
        raise AgentError("Invalid user role.")

    normalized_role = WEBSITE_ROLE_MAP.get(
        role.strip().lower()
    )

    if normalized_role is None:
        raise AgentError("Unauthorized user role.")

    return normalized_role


def validate_subject(subject):

    if not isinstance(subject, str):
        raise AgentError("Invalid user identity.")

    if not subject.strip():
        raise AgentError("User identity is required.")

    return subject.strip()


def validate_website_answer(answer):

    if not isinstance(answer, str):
        raise AgentError("Invalid AI response type.")

    answer = answer.strip()

    invalid_responses = {
        "",
        "none",
        "null",
        "undefined",
        "no answer returned",
        "no response",
        "empty response"
    }

    normalized = (
        answer.lower().rstrip(".!?").strip()
    )

    if normalized in invalid_responses:

        raise AgentError(
            "AI did not produce a valid final answer."
        )

    return answer


# Website AI chat interface
def eye_style_chat(
    question,
    workflow,
    authenticated_role,
    authenticated_subject
):

    # IMPORTANT:
    # Role and identity must come from a trusted backend
    # after Firebase token verification.
    # Never trust role values supplied by a browser.

    if not isinstance(question, str):
        raise AgentError("Question must be text.")

    question = question.strip()

    if not question:
        raise AgentError("Question cannot be empty.")

    if len(question) > MAX_QUESTION_LENGTH:

        raise AgentError(
            "Question exceeds length limit."
        )

    role = normalize_website_role(
        authenticated_role
    )

    subject = validate_subject(
        authenticated_subject
    )

    if not isinstance(workflow, str):
        raise AgentError("Invalid workflow.")

    # Daniel's RAG knowledge assistant
    # Reuse the existing initialized RAG implementation.
    # Do not create another embedding model or vector index.

    if workflow == "RAG":

        if not callable(globals().get("ask_rag")):

            raise AgentError(
                "RAG service is not initialized."
            )

        result = ask_rag(
            question=question,
            authenticated_role=role,
            authenticated_subject=subject
        )

        if not isinstance(result, dict):

            raise AgentError("Invalid RAG response.")

        if result.get("success") is not True:

            raise AgentError("RAG request failed.")

        answer = validate_website_answer(
            result.get("answer")
        )

        # Return only website-facing response fields.
        # Preserve citations and abstention information
        # without exposing complete retrieved passages.

        response = {
            "success": True,
            "answer": answer,
            "workflow": "RAG",
            "role": role,
            "reports": [],
            "sources": result.get("sources", []),
            "notices": result.get("notices", []),
            "abstained": result.get("abstained", False),
            "top_similarity": result.get(
                "top_similarity"
            ),
            "mode": "demo",
            "requires_staff_review": True
        }

        return json_compatible(response)

    # Existing operational workflow permissions

    if workflow not in ROLE_WORKFLOWS.get(role, set()):

        raise AgentError(
            "Access to this workflow is denied."
        )

    specialist, restricted_gateway = (
        create_workflow_agent(
            staff_role=role,
            workflow=workflow,
            subject=subject
        )
    )

    agent_question = (
        question
        + "\nUse authorized data only. "
        + "Do not modify records or contact patients. "
        + "Describe model outputs as risk scores, "
        + "not calibrated probabilities. "
        + "Return a final written answer after "
        + "retrieving sufficient information."
    )

    result = specialist.ask(agent_question)

    if not isinstance(result, dict):
        raise AgentError("Invalid agent response.")

    answer = validate_website_answer(
        result.get("answer")
    )

    sources = list(
        restricted_gateway.read_trace
    )

    notices = []

    if "GetNoShowRisk" in sources:

        notices.append(RISK_SCORE_NOTICE)

    response = {
        "success": True,
        "answer": answer,
        "workflow": workflow,
        "role": role,
        "reports": result.get("reports", []),
        "sources": sources,
        "notices": notices,
        "mode": "demo",
        "requires_staff_review": True
    }

    return json_compatible(response)


# Website dashboard interface
def eye_style_dashboard(
    authenticated_role,
    authenticated_subject
):

    role = normalize_website_role(
        authenticated_role
    )

    subject = validate_subject(
        authenticated_subject
    )

    records = get_dashboard_records(
        staff_role=role,
        subject=subject
    )

    allowed_workflows = ROLE_WORKFLOWS[role]

    # Verify dashboard permissions and output structure
    for record in records:

        if record["workflow_type"] not in allowed_workflows:

            raise AgentError(
                "Unauthorized dashboard record."
            )

        validation = validate_dashboard_output(
            record
        )

        if not validation["valid"]:

            raise AgentError(
                "Dashboard output failed validation."
            )

    response = {
        "success": True,
        "mode": "demo",
        "role": role,
        "record_count": int(len(records)),
        "records": records
    }

    return json_compatible(response)


# Website capabilities interface
def eye_style_capabilities(
    authenticated_role,
    authenticated_subject
):

    role = normalize_website_role(
        authenticated_role
    )

    validate_subject(authenticated_subject)

    # RAG availability is determined from the
    # existing initialized RAG function.

    rag_available = callable(
        globals().get("ask_rag")
    )

    available_workflows = set(
        ROLE_WORKFLOWS[role]
    )

    if rag_available:

        available_workflows.add("RAG")

    return {
        "success": True,
        "role": role,
        "mode": "demo",
        "available_workflows": sorted(
            available_workflows
        ),
        "read_only": True,
        "live_crystal_connected": False,
        "rag_available": rag_available,
        "automatic_patient_contact": False,
        "model_score_calibrated": False
    }


print("=" * 60)
print("EYE STYLE WEBSITE BACKEND INTERFACE")
print("=" * 60)

print("AI chat interface: Ready")
print("Dashboard interface: Ready")
print("Capabilities interface: Ready")
print("Role mapping: Ready")
print("JSON conversion: Ready")
print("Response validation: Ready")
print("Data mode: Synthetic demo")

print(
    "RAG integration:",
    "Ready" if callable(
        globals().get("ask_rag")
    ) else "Not initialized"
)

print("HTTP deployment: Pending")
print("Firebase authentication: Pending")

print("\nWebsite backend functions initialized.")